In [1]:
!pip install -q -U transformers datasets accelerate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [48]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from datasets import Dataset

BERT com arquitetura atualizada (RoPE, GeGLU, FlashAttention)

In [49]:
import os
import getpass

os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face token: ")
hf_token = os.environ["HF_TOKEN"]

Hugging Face token: ··········


In [59]:
MODEL_ID = "neuralmind/bert-base-portuguese-cased"
#tokenizer = AutoTokenizer.from_pretrained(MODEL_ID) # opção sem toker, pode ser mais lento ao baixar o BERT no HF
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)

In [60]:
# ==============================================================================
# PARTE 1: INSPECIONANDO A CABEÇA ORIGINAL (Masked LM Head)
# ==============================================================================
print("--- 1. MODELO ORIGINAL (PREVISÃO DE PALAVRAS) ---")
mlm_model = AutoModelForMaskedLM.from_pretrained(MODEL_ID)
cabeca_original = mlm_model.get_output_embeddings()

print("Cabeça original (decoder):", cabeca_original)
print("Saída (tamanho do vocabulário):", cabeca_original.out_features)

# Liberando memória antes de carregar a versão modificada
del mlm_model
torch.cuda.empty_cache()

--- 1. MODELO ORIGINAL (PREVISÃO DE PALAVRAS) ---


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cabeça original (decoder): Linear(in_features=768, out_features=29794, bias=True)
Saída (tamanho do vocabulário): 29794


In [61]:
# ==============================================================================
# PARTE 2: TROCANDO A CABEÇA PARA CLASSIFICAÇÃO (2 CLASSES)
# ==============================================================================
print("\n--- 2. TROCANDO A CABEÇA PARA CLASSIFICAÇÃO ---")
# O Hugging Face descarta o 'decoder' de 50.368 saídas e instala um 'classifier' de 2 saídas
clf_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    id2label={0: "Negativo", 1: "Positivo"},
    label2id={"Negativo": 0, "Positivo": 1}
)


# ==============================================================================
# CONGELANDO O BACKBONE (PARA TREINAR APENAS A NOVA CABEÇA)
# ==============================================================================
# No ModernBERT, o corpo principal fica dentro do atributo '.model'
for param in clf_model.bert.parameters():
    param.requires_grad = False

# # Descongela APENAS a última camada (camada 11) + a camada pooler
# for param in clf_model.bert.encoder.layer[-1].parameters():
#     param.requires_grad = True
# for param in clf_model.bert.pooler.parameters():
#     param.requires_grad = True

# Verificação prática da economia de treinamento
total_params = sum(p.numel() for p in clf_model.parameters())
treinaveis = sum(p.numel() for p in clf_model.parameters() if p.requires_grad)

print(f"Total de parâmetros: {total_params:,}")
print(f"Parâmetros treináveis (apenas cabeça): {treinaveis:,} ({treinaveis/total_params:.4%})")

print("Nova cabeça instalada (classifier):", clf_model.classifier)


--- 2. TROCANDO A CABEÇA PARA CLASSIFICAÇÃO ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

Total de parâmetros: 108,924,674
Parâmetros treináveis (apenas cabeça): 1,538 (0.0014%)
Nova cabeça instalada (classifier): Linear(in_features=768, out_features=2, bias=True)


In [62]:
# ==============================================================================
# PARTE 3: TREINANDO A NOVA CABEÇA (FINE-TUNING RÁPIDO)
# ==============================================================================
# Criando um pequeno dataset de exemplo em Português
dados = {
    "text": [
        "O produto chegou antes do prazo e a qualidade é excelente!",
        "Adorei o atendimento, super recomendo a loja.",
        "Funciona perfeitamente, superou minhas expectativas.",
        "Melhor compra que fiz este ano, nota 10.",
        "Péssima experiência, o item veio quebrado e ninguém responde.",
        "Não recomendo, dinheiro jogado fora. Muito frágil.",
        "Atrasou duas semanas e o suporte foi extremamente grosso.",
        "Horrível, parou de funcionar no segundo dia de uso."
    ],
    "label": [1, 1, 1, 1, 0, 0, 0, 0] # 1 = Positivo, 0 = Negativo
}



dataset = Dataset.from_dict(dados)

dataset_tokenizado = dataset.map(
    lambda batch: tokenizer(batch["text"], padding=True, truncation=True, max_length=128),
    batched=True
)
#def tokenizar(batch):
#    return tokenizer(batch["text"], padding=True, truncation=True, max_length=128)

#dataset_tokenizado = dataset.map(tokenizar, batched=True)


# training_args = TrainingArguments(
#     output_dir="./modernbert-otimizado",
#     num_train_epochs=25,              # 50 passos totais
#     per_device_train_batch_size=4,
#     learning_rate=2e-3,               # Taxa alta segura para treinar só a cabeça
#     warmup_steps=0,                 # Sem aquecimento lento no início
#     weight_decay=0.0,                 # Permite convergência máxima
#     logging_steps=10,                 # Mostra a queda da loss a cada 10 passos
#     fp16=True,
#     report_to="none"
# )
training_args = TrainingArguments(
    output_dir="./bertimbau-congelado",
    num_train_epochs=30,              # 30 passos (suficiente sem decorar)
    per_device_train_batch_size=4,
    learning_rate=2e-2,               # Passo moderado para a cabeça
    weight_decay=0.01,                # Regularização para não decorar ruído
    logging_steps=10,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=clf_model,
    args=training_args,
    train_dataset=dataset_tokenizado,
)

print("\n--- 3. INICIANDO TREINAMENTO DA NOVA CABEÇA ---")
trainer.train()

Map:   0%|          | 0/8 [00:00<?, ? examples/s]


--- 3. INICIANDO TREINAMENTO DA NOVA CABEÇA ---


Step,Training Loss
10,0.805801
20,0.372715
30,0.049588
40,0.085786
50,0.054338
60,0.030107


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=60, training_loss=0.2330557331442833, metrics={'train_runtime': 16.0769, 'train_samples_per_second': 14.928, 'train_steps_per_second': 3.732, 'total_flos': 1973332915200.0, 'train_loss': 0.2330557331442833, 'epoch': 30.0})

In [63]:
# ==============================================================================
# PARTE 4: TESTANDO O MODELO COM A NOVA CABEÇA TREINADA
# ==============================================================================
print("\n--- 4. TESTE DE INFERÊNCIA ---")
frases_teste = [
    "Estou muito satisfeito com o resultado, ficou ótimo!",
    "Que decepção, não funciona de jeito nenhum.",
    "Entrega rápida, mas o material é bem resistente e bonito.",
    "Não vale o preço, me arrependi da compra."
]

inputs = tokenizer(frases_teste, padding=True, truncation=True, return_tensors="pt").to("cuda")

clf_model.eval()
with torch.no_grad():
    logits = clf_model(**inputs).logits
    probabilidades = torch.softmax(logits, dim=-1)

for frase, probs in zip(frases_teste, probabilidades):
    classe_predita = torch.argmax(probs).item()
    label = clf_model.config.id2label[classe_predita]
    confianca = probs[classe_predita].item() * 100
    print(f"Frase: '{frase}'\n -> Previsão: {label} ({confianca:.1f}% de confiança)\n")



--- 4. TESTE DE INFERÊNCIA ---
Frase: 'Estou muito satisfeito com o resultado, ficou ótimo!'
 -> Previsão: Positivo (76.8% de confiança)

Frase: 'Que decepção, não funciona de jeito nenhum.'
 -> Previsão: Negativo (89.9% de confiança)

Frase: 'Entrega rápida, mas o material é bem resistente e bonito.'
 -> Previsão: Positivo (63.9% de confiança)

Frase: 'Não vale o preço, me arrependi da compra.'
 -> Previsão: Negativo (59.5% de confiança)

